<a href="https://colab.research.google.com/github/matchapresso/AI-Agent-with-RAG/blob/main/2.%20AI%20Agent%20with%20Tool%20Use.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 2: AI Agent with Tool Use & RAG

**Author:** Areta Vahtsa Nur Kirana
**Date:** 5 January 2026

## 1. Project Overview
This notebook implements an autonomous AI Agent capable of reasoning and using external tools to solve complex queries. The agent is built using **LangGraph** (for orchestration) and **Ollama** (running **Qwen 2.5**).

## 2. Technical Architecture
* **LLM Engine:** Qwen 2.5 (7B) - Chosen for its superior tool-calling capabilities compared to standard Llama 3 models.
* **Orchestrator:** LangGraph StateGraph (ReAct Architecture).
* **Knowledge Base:** A Hybrid RAG pipeline (BM25 + ChromaDB) indexing the "Attention Is All You Need" paper.
* **Tools:**
    * `rag_search`: Technical retrieval tool.
    * `calculator`: Safe mathematical evaluator.
    * `summarizer`: Text condensation tool.
    * `text_transformer`: String manipulation utility.

In [ ]:
!nvidia-smi #Test the GPU

## 3. Environment Setup & Dependencies

The necessary libraries are installed below to support the agentic workflow. Key dependencies include:
* **LangChain & LangGraph:** Frameworks for building the agent's cognitive architecture.
* **ChromaDB:** A vector database used for storing embeddings of the research paper.
* **Arxiv & PyMuPDF:** Utilities to download and parse technical documentation.
* **Rank_BM25:** A library enabling keyword-based sparse retrieval (Hybrid Search).

*Note: System-level fixes (`pciutils`) are included to ensure the Kaggle T4 GPU is correctly detected by Ollama.*

In [ ]:
# 1. Environment Setup
# Installing necessary packages for the RAG pipeline and Agent graph
import sys
import os

print("⚙️ Setting up environment...")

# Install dependencies (Quiet mode to reduce log clutter)
!pip install -q langchain langchain-community langchain-chroma langchain-ollama langgraph arxiv pymupdf rank_bm25 sentence-transformers
# Run the command below to fix for Kaggle GPU detection
#!apt-get update -y && apt-get install -y pciutils


In [ ]:
# Imports
import operator
import subprocess
import time
from typing import Annotated, List, TypedDict, Union

# LangChain and LangGraph import
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

## 4. Model Initialization (Qwen 2.5)

**Qwen 2.5 (7B)** is utilized as the core reasoning engine.

**Architectural Decision:**
Qwen 2.5 was selected for this task due to its superior performance in **Tool Calling** and **Structured Output** generation within the 7B parameter class. This ensures reliable formatting of tool inputs (e.g., passing correct arguments to the calculator).

The code below initializes the Ollama server in the background and pulls the quantized model to the local environment.

In [ ]:
def setup_ollama_server():
    print("⏳ Starting Ollama Server...")
    # Start server in background
    process = subprocess.Popen("ollama serve", shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(10) # Give it time to spin up

    print("⏳ Pulling Qwen 2.5 Model...")
    !ollama pull qwen2.5
    print("Model Ready.")

setup_ollama_server()

# Global LLM instance for consistent usage. Temperature=0 ensures deterministic/factual outputs for the RAG tools
llm_engine = ChatOllama(model="qwen2.5", temperature=0)

## 5. RAG Pipeline Construction (The "Knowledge Base")

To satisfy the requirement of answering technical questions about the *"Attention Is All You Need"* paper, a specialized **Hybrid Retrieval Pipeline** is constructed.

**Pipeline Steps:**
1.  **Ingestion:** The paper (Arxiv: 1706.03762) is downloaded dynamically.
2.  **Chunking:** Text is split into 512-character chunks with overlap to preserve context.
3.  **Embedding:** `BAAI/bge-base-en-v1.5` (running on GPU) is used to convert text into vector representations.
4.  **Indexing (Ensemble):**
    * **Dense Index (Chroma):** Captures semantic meaning (e.g., "model size" $\approx$ "d_model").
    * **Sparse Index (BM25):** Captures exact keyword matches.
    * **Ensemble:** Both retrieval methods are weighted 50/50 to optimize accuracy.

In [ ]:
# 3. KNOWLEDGE BASE (RAG PIPELINE)

# Global variable to store the retriever engine
RAG_ENGINE = None

def build_knowledge_base():
    """
    Builds a Hybrid Search Index (Vector + Keyword) for the Agent.
    Paper: 'Attention Is All You Need' (Arxiv: 1706.03762)
    """
    global RAG_ENGINE
    print("Building Knowledge Base...")

    # A. Load Data
    # We load the specific paper required for the assignment
    loader = ArxivLoader(query="1706.03762", load_max_docs=1)
    raw_docs = loader.load()

    # B. Split Text
    # 512 chars is optimal for preserving context without overloading the LLM
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
    chunks = text_splitter.split_documents(raw_docs)

    # C. Embeddings (GPU Accelerated)
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-base-en-v1.5",
        model_kwargs={'device': 'cuda'}
    )

    # D. Vector Store (Dense Retrieval)
    vectorstore = Chroma.from_documents(chunks, embeddings, collection_name="agent_rag_db")
    dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # E. Keyword Search (Sparse Retrieval)
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 3

    # F. Ensemble Retriever
    # Weights are 50/50 to balance semantic understanding with exact keyword matching
    RAG_ENGINE = EnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=[0.5, 0.5]
    )
    print("Knowledge Base Built successfully.")

# Initialize immediately
build_knowledge_base()

## 6. Agent Architecture & Tool Definitions

The cognitive architecture of the agent is defined below using **LangGraph**.

### A. The Tools
Four distinct tools are implemented, each decorated with `@tool`. Detailed docstrings are provided to serve as the "System Prompt" for the LLM, instructing it on *when* and *how* to use each tool.
* **`rag_search`**: Connects the agent to the RAG pipeline built in Step 5.
* **`calculator`**: A safe Python `eval()` wrapper for mathematical operations.
* **`summarizer`**: A recursive call to the LLM for text condensation.
* **`text_transformer`**: A utility function for string operations.

### B. The Graph Structure
A cyclic graph is implemented with two primary nodes:
1.  **`agent` Node:** The LLM receives the user query and history, deciding whether to generate an answer or call a tool.
2.  **`tools` Node:** Executes the requested tool and returns the output to the agent.
3.  **`router` Edge:** A conditional edge that loops the conversation until the agent determines the task is complete (`END`).

In [ ]:
# --- TOOL DEFINITIONS ---
@tool
def rag_search(query: str) -> str:
    """
    Search for technical details about the 'Attention Is All You Need' paper,
    Transformer architecture, or self-attention mechanisms.
    """
    if not RAG_ENGINE:
        return "Error: Database not ready."

    docs = RAG_ENGINE.invoke(query)
    # Return source page numbers for traceability
    return "\n\n".join([f"[Source: Page {d.metadata.get('page','?')}] {d.page_content}" for d in docs])

@tool
def calculator(expression: str) -> str:
    """
    Calculates mathematical expressions.
    Input examples: '200 / 5' or '10 + 5 * 2'.
    """
    try:
        # Restricted environment for safety
        allowed_names = {"abs": abs, "min": min, "max": max, "round": round}
        return str(eval(expression, {"__builtins__": None}, allowed_names))
    except Exception as e:
        return f"Math Error: {e}"

@tool
def summarizer(text: str) -> str:
    """Summarizes a long piece of text into a concise paragraph."""
    # Reuse Qwen 2.5 for summarization tasks
    return llm_engine.invoke(f"Summarize this text in 2 sentences:\n{text}").content

@tool
def text_transformer(text: str, operation: str) -> str:
    """
    Transforms text strings.
    Valid operations: 'upper' (uppercase), 'lower' (lowercase), 'reverse' (reverses text).
    """
    if operation == "upper": return text.upper()
    elif operation == "lower": return text.lower()
    elif operation == "reverse": return text[::-1]
    return "Error: Unknown operation"

# --- GRAPH CONSTRUCTION ---

# 1. Bind tools to the LLM
tools = [rag_search, calculator, summarizer, text_transformer]
llm_with_tools = llm_engine.bind_tools(tools)

# 2. Define State
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

# 3. Define Nodes
def agent_node(state):
    """The reasoning engine (LLM)"""
    return {"messages": [llm_with_tools.invoke(state['messages'])]}

def router(state):
    """Decides next step: Tool or End?"""
    if state['messages'][-1].tool_calls:
        return "tools"
    return END

# 4. Build Graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(tools))

workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", router, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")

app = workflow.compile()
print("Agent Architecture Compiled.")

## 7. Execution & Evaluation

To validate the agent's capabilities, a test suite is executed covering the five distinct scenarios required by the problem statement.

**Test Cases:**
1.  **RAG Retrieval:** Verifies the ability to query the vector database for specific facts ("d_model").
2.  **Math Capability:** Tests the tool-calling logic for the `calculator`.
3.  **Summarization:** Validates the handling of text processing tasks.
4.  **Multi-step Logic:** A complex query requiring the **retrieval** of a value (RAG) followed by **processing** (Math), demonstrating "Reasoning Chains."
5.  **Text Transformation:** Verifies the usage of the additional utility tool.

In [ ]:
# 5. EVALUATION & TESTING
def run_test_case(scenario_name, query, run_id):
    print(f"\n{'='*15} TEST CASE #{run_id}: {scenario_name} {'='*15}")
    print(f"🗣️ User Query: \"{query}\"")
    print("-" * 50)

    state = {"messages": [HumanMessage(content=query)]}

    try:
        # Stream the agent's thought process
        for event in app.stream(state):
            if "agent" in event:
                msg = event["agent"]["messages"][0]
                if msg.tool_calls:
                    print(f"Agent Decision: Using Tool -> {msg.tool_calls[0]['name']}")
                    print(f"   Input: {msg.tool_calls[0]['args']}")
                else:
                    print(f"Agent Final Answer:\n{msg.content}")
            elif "tools" in event:
                # Preview tool output (truncated if too long)
                output = event["tools"]["messages"][0].content
                preview = (output[:100] + '...') if len(output) > 100 else output
                print(f"Tool Output: {preview}")

    except Exception as e:
        print(f"Error during execution: {e}")

In [ ]:
# --- EXECUTE REQUIRED SCENARIOS ---

# 1. RAG Retrieval
run_test_case("RAG Retrieval", "What is the d_model dimension mentioned in the paper?", 1)

# 2. Math Capability
run_test_case("Math Operation", "Calculate 500 divided by 4, then add 25.", 2)

# 3. Summarization
run_test_case("Summarization", "Summarize this: 'The Transformer is a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output.'", 3)

# 4. Multi-step Logic (RAG + Math)
run_test_case("Multi-step Logic", "Find the d_model dimension from the paper and divide it by 8.", 4)

# 5. Text Transformation
run_test_case("Text Transformation", "Reverse the word 'DataScience'.", 5)